# Northward Heat Transport

Depth-integrated meridional (northward) heat transport from MOM6, decomposed
by ocean basin (Global, Atlantic, Indo-Pacific). Compared against Trenberth &
Caron reanalysis and Ganachaud & Wunsch observational estimates.

Authors: John Krasting

In [ ]:
# Development mode: constantly refreshes module code
%load_ext autoreload
%autoreload 2

## Framework Code and Diagnostic Setup

In [ ]:
import esnb
from esnb import NotebookDiagnostic, RequestedVariable, CaseGroup2, nbtools
from esnb.sites.gfdl import call_dmget, convert_to_momgrid
import climplot

In [ ]:
# Define requested variable: depth-integrated northward heat transport
# List realm allows fallback from ocean_annual (ESM4.5) to ocean_monthly (CM4.0)
variables = [
    RequestedVariable("T_ady_2d", ["ocean_annual", "ocean_monthly"], frequency="yearly"),
]

# Diagnostic configuration
diag_name = "Northward Heat Transport"
diag_desc = "Depth-integrated meridional heat transport by ocean basin"
user_options = {}

# Initialize the diagnostic
diag = NotebookDiagnostic(diag_name, diag_desc, variables=variables, **user_options)

In [ ]:
# Define experiments to analyze
groups = [
    CaseGroup2("odiv-1", date_range=("0401-01-01", "0500-12-31"), name="CM4.0"),
    CaseGroup2("esm45-122", date_range=("0401-01-01", "0500-12-31"), name="ESM4.5 Spinup"),
    CaseGroup2("esm45-137", date_range=("0001-01-01", "0060-12-31"), name="ESM4.5 piControl"),
    CaseGroup2("esm45-148", date_range=("0001-01-01", "0020-12-31"), name="ESM4.5 piControl v2"),
]

In [ ]:
# Combine experiments with the diagnostic request and determine files to load
diag.resolve(groups)

In [ ]:
# Print a list of file paths
_ = [print(x) for x in diag.files]

*(The files above are necessary to run the diagnostic.)*

In [ ]:
# Stage files from mass storage
call_dmget(diag.files, status=True)
call_dmget(diag.files)

In [ ]:
# Load the data as xarray datasets
diag.open()

In [ ]:
# Add MOM6 grid metadata (no corners needed for line plots)
convert_to_momgrid(diag)

## Main Diagnostic

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import cmip_basins

In [ ]:
all_figs = []
climplot.publication()

# Load Trenberth & Caron reanalysis heat transport
dsobs = xr.open_dataset(
    "/archive/jpk/om4labs/datasets/Trenberth_and_Caron/"
    "Trenberth_and_Caron_Heat_Transport.nc"
)

yobs = dsobs.ylat.values
NCEP_Global = dsobs.OTn.values
NCEP_Atlantic = dsobs.ATLn.values
NCEP_IndoPac = dsobs.INDPACn.values
ECMWF_Global = dsobs.OTe.values
ECMWF_Atlantic = dsobs.ATLe.values
ECMWF_IndoPac = dsobs.INDPACe.values

In [ ]:
class GWObs:
    """Ganachaud & Wunsch (2003) observational heat transport estimates."""

    class _gw:
        def __init__(self, lat, trans, err):
            self.lat = lat
            self.trans = trans
            self.err = err
            self.minerr = trans - err
            self.maxerr = trans + err

        def annotate(self, ax):
            for n in range(0, len(self.minerr)):
                if n == 0:
                    ax.plot(
                        [self.lat[n], self.lat[n]],
                        [self.minerr[n], self.maxerr[n]],
                        "c",
                        linewidth=2.0,
                        label="G&W",
                    )
                else:
                    ax.plot(
                        [self.lat[n], self.lat[n]],
                        [self.minerr[n], self.maxerr[n]],
                        "c",
                        linewidth=2.0,
                    )
                ax.scatter(self.lat, self.trans, marker="s", facecolor="cyan")

    def __init__(self):
        self.gbl = self._gw(
            np.array([-30.0, -19.0, 24.0, 47.0]),
            np.array([-0.6, -0.8, 1.8, 0.6]),
            np.array([0.3, 0.6, 0.3, 0.1]),
        )
        self.atl = self._gw(
            np.array([-45.0, -30.0, -19.0, -11.0, -4.5, 7.5, 24.0, 47.0]),
            np.array([0.66, 0.35, 0.77, 0.9, 1.0, 1.26, 1.27, 0.6]),
            np.array([0.12, 0.15, 0.2, 0.4, 0.55, 0.31, 0.15, 0.09]),
        )
        self.indpac = self._gw(
            np.array([-30.0, -18.0, 24.0, 47.0]),
            np.array([-0.9, -1.6, 0.52, 0.0]),
            np.array([0.3, 0.6, 0.2, 0.05]),
        )

GW = GWObs()

In [ ]:
# Compute depth-integrated northward heat transport by basin
varobj = diag.varmap["T_ady_2d"]

gbl_results = []
atl_results = []
pac_results = []

for group in diag.groups:
    ds = group.datasets[varobj]
    data = ds["T_ady_2d"].mean("time")
    lat = ds.geolat_v.mean("xh")

    # Global: sum over all longitudes, convert to PW
    gbl = data.sum("xh") * 1.0e-15
    gbl = gbl.assign_coords({"yq": lat.values}).rename({"yq": "lat"})
    gbl_results.append(gbl.load())

    # Basin masks using cmip_basins on the v-grid
    mask = cmip_basins.generate_basin_codes(ds, lon="geolon_v", lat="geolat_v")

    # Atlantic: basin codes 2, 4, 6, 7, 8, 9
    atl_mask = xr.where(
        (mask == 2) | (mask == 4) | (mask == 6)
        | (mask == 7) | (mask == 8) | (mask == 9),
        1, 0,
    )
    atl = (data * atl_mask).sum("xh") * 1.0e-15
    atl = atl.assign_coords({"yq": lat.values}).rename({"yq": "lat"})
    atl_results.append(atl.load())

    # Indo-Pacific: basin codes 3, 5
    pac_mask = xr.where((mask == 3) | (mask == 5), 1, 0)
    pac = (data * pac_mask).sum("xh") * 1.0e-15
    pac = pac.assign_coords({"yq": lat.values}).rename({"yq": "lat"})
    pac_results.append(pac.load())

In [ ]:
# Extract max/min heat transport and latitudes per basin per group
for n, group in enumerate(diag.groups):
    for basin_name, results in [
        ("nht_global", gbl_results),
        ("nht_atlantic", atl_results),
        ("nht_indopac", pac_results),
    ]:
        vals = results[n].fillna(0.0).values
        lat_vals = results[n].lat.fillna(0.0).values

        max_idx = np.argmax(vals)
        min_idx = np.argmin(vals)

        group.add_metric(basin_name, ("max_pw", round(float(vals[max_idx]), 4)))
        group.add_metric(basin_name, ("max_lat", round(float(lat_vals[max_idx]), 2)))
        group.add_metric(basin_name, ("min_pw", round(float(vals[min_idx]), 4)))
        group.add_metric(basin_name, ("min_lat", round(float(lat_vals[min_idx]), 2)))

In [ ]:
# Citation strings for figure annotations
cite_tc = (
    r"Trenberth, K. E. and J. M. Caron, 2001: Estimates of Meridional "
    r"Atmosphere and Ocean Heat Transports. J.Climate, 14, 3433-3443."
)
cite_gw1 = (
    r"Ganachaud, A. and C. Wunsch, 2000: Improved estimates of global "
    r"ocean circulation, heat transport and mixing from hydrographic data."
)
cite_gw2 = r"Nature, 408, 453-457"


def plot_nht(ax, fig, results, ncep, ecmwf, gw, title, xlim=None, ylim=None):
    """Plot northward heat transport for one basin."""
    for n, group in enumerate(diag.groups):
        ax.plot(results[n].lat, results[n].values, linewidth=1.0, label=group.name)
    ax.plot(yobs, ncep, "k--", linewidth=0.3, label="NCEP")
    ax.plot(yobs, ecmwf, "k.", linewidth=0.3, label="ECMWF")
    gw.annotate(ax)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(True, linestyle=":")
    ax.legend(fontsize=6)
    fig.text(0.5, 0.9, title, ha="center", fontsize=10)
    fig.text(0.05, 0.00, cite_tc, fontsize=6)
    fig.text(0.05, -0.03, cite_gw1, fontsize=6)
    fig.text(0.05, -0.06, cite_gw2, fontsize=6)

In [ ]:
# Global Northward Heat Transport
fig = plt.figure(figsize=(6, 3.5))
ax = plt.subplot(1, 1, 1)
plot_nht(ax, fig, gbl_results, NCEP_Global, ECMWF_Global, GW.gbl,
         "Global", ylim=(-2, 2.5))
all_figs.append(fig)

In [ ]:
# Atlantic Northward Heat Transport
fig = plt.figure(figsize=(6, 3.5))
ax = plt.subplot(1, 1, 1)
plot_nht(ax, fig, atl_results, NCEP_Atlantic, ECMWF_Atlantic, GW.atl,
         "Atlantic", xlim=(-30, 80), ylim=(-0.05, 1.8))
all_figs.append(fig)

In [ ]:
# Indo-Pacific Northward Heat Transport
fig = plt.figure(figsize=(6, 3.5))
ax = plt.subplot(1, 1, 1)
plot_nht(ax, fig, pac_results, NCEP_IndoPac, ECMWF_IndoPac, GW.indpac,
         "Pacific", xlim=(-34, 80), ylim=(-2.5, 1.5))
all_figs.append(fig)

## Write Metrics

In [ ]:
diag.write_metrics("NHT_metrics.json")

## Save PowerPoint

In [ ]:
nbtools.save_pptx(all_figs, "northward_heat_transport.pptx")